# Khmer TTS fine-tuning (T4, resumable)

Runtime > Change runtime type > T4 GPU, before running anything.

Order: install -> login -> inspect datasets -> prepare merged dataset -> build train-ready checkpoint -> edit config -> run resumable training loop.

In [ ]:
# 1. Install dependencies
!pip install -q datasets soundfile librosa huggingface_hub accelerate
!git clone -q https://github.com/ylacombe/finetune-hf-vits.git
!pip install -q -r finetune-hf-vits/requirements.txt
!cd finetune-hf-vits/monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace

In [ ]:
# 2. Log in with a WRITE token (needed to push checkpoints)
from huggingface_hub import login
login()

## 3. Upload the pipeline scripts
Upload `inspect_datasets.py`, `prepare_dataset.py`, `training_config.json`, and `run_training_colab.sh` into this Colab session (drag into the Files pane, or `git clone` your own repo if you've committed them there).

In [ ]:
# 4. Inspect the source datasets BEFORE deciding a speaker strategy.
# Read the output carefully -- this decides how you configure
# SPEAKER_STRATEGY / FORCE_SINGLE_SPEAKER in prepare_dataset.py.
!python inspect_datasets.py

Edit `prepare_dataset.py`'s `SPEAKER_STRATEGY` / `FORCE_SINGLE_SPEAKER` now, based on the inspection output above, before running the next cell.

In [ ]:
# 5. Merge + clean the 12 datasets, push the training-ready dataset.
!python prepare_dataset.py

In [ ]:
# 6. One-time: build a training-ready (generator+discriminator) checkpoint.
# See setup_train_checkpoint.md for the reasoning.
!python finetune-hf-vits/convert_original_discriminator_checkpoint.py \
    --language_code khm \
    --pytorch_dump_folder_path ./khm-train-ready \
    --push_to_hub phonsobon/mms-tts-khm-train-ready

Now double-check `training_config.json`: `dataset_name`, `model_name_or_path`, `hub_model_id` should point where you want. Also worth confirming the actual resume flag name for your checked-out repo version:

In [ ]:
!python finetune-hf-vits/run_vits_finetuning.py --help | grep -i -E 'resum|checkpoint'

In [ ]:
# 7. Launch the crash-resilient, auto-resuming training loop.
# If Colab disconnects: reconnect, re-run this same cell -- it will
# pull the latest checkpoint pushed to the Hub and continue.
!chmod +x run_training_colab.sh
!bash run_training_colab.sh